In [42]:
# Cài thư viện
import osmnx as ox #open street map: mô hình hóa và trực quan hóa mạng lưới giao thông
import requests # gọi api
import pandas as pd # Xử lý phân tích và làm sạch dữ liệu
import numpy as np # tính toán
import time # thời gian
from datetime import datetime, timedelta
import networkx as nx # thư viện dùng để tạo, nghiên cứu cấu trúc của grap
import folium


In [16]:
CENTER_POINT = (21.0025, 105.8037) 
# Bán kính 3km (đủ bao trùm Vành đai 3 và các đường lân cận như Nguyễn Trãi, Khuất Duy Tiến)
DISTANCE_M = 3000
# Bỏ qua đường dân sinh, đường nội bộ chỉ lấy cao tốc, quốc lộ, đường trục, đườn cấp 1, cấp 2
cf = '["highway"~"motorway|trunk|primary|secondary"]'
# Cấu hình thời gian (7 ngày, 5 phút/lần)
DAYS = 1
FREQ_MIN = 5
TOTAL_STEPS = (24 * 60 // FREQ_MIN) * DAYS # 2016 bước thời gian
MAX_EDGES_TO_SAMPLE = 1000 # giới hạn 1000 cạnh
print(f"🚀 Đang khởi tạo quy trình sinh dữ liệu cho {DAYS} ngày ({TOTAL_STEPS} bước)...")


🚀 Đang khởi tạo quy trình sinh dữ liệu cho 1 ngày (288 bước)...


In [26]:
G = ox.graph_from_point(
    CENTER_POINT, 
    dist=DISTANCE_M, 
    network_type="drive",
    custom_filter=cf
) # lấy grap
# Lấy danh sách các Nút (Nodes) - Đây chính là N trong mô hình T-GCN
nodes_gdf = ox.graph_to_gdfs(G, edges=False, nodes=True)
node_ids = nodes_gdf.index.tolist()
num_nodes = len(node_ids)

# Tạo Ma trận Kề (Adjacency Matrix)
adj_matrix = nx.adjacency_matrix(G).todense()

print(f"✅ Đã tạo Graph thành công!")
print(f"   - Số lượng Nút (Nodes/Sensors): {num_nodes}")
print(f"   - Kích thước Ma trận kề: {adj_matrix.shape}")

✅ Đã tạo Graph thành công!
   - Số lượng Nút (Nodes/Sensors): 624
   - Kích thước Ma trận kề: (624, 624)


In [19]:
# 3. SINH DỮ LIỆU MÔ PHỎNG (TEMPORAL)
# ==========================================
print("🔄 Đang sinh dữ liệu Tốc độ, Lưu lượng và Thời tiết...")

# Tạo trục thời gian
start_time = datetime(2024, 12, 1, 0, 0, 0)
time_index = pd.date_range(start=start_time, periods=TOTAL_STEPS, freq=f'{FREQ_MIN}min')

🔄 Đang sinh dữ liệu Tốc độ, Lưu lượng và Thời tiết...


In [22]:
# --- Hàm mô phỏng mẫu hình giao thông ---
def generate_traffic_pattern(n_steps, n_nodes):
    # 1. Tạo mẫu hình ngày (Daily Pattern): Giờ cao điểm sáng (7-9h) và chiều (17-19h)
    # Sử dụng sóng sin kết hợp để tạo 2 đỉnh tắc nghẽn
    x = np.linspace(0, 2 * np.pi * DAYS, n_steps)
    
    # Mẫu hình cơ bản: Cao vào ban đêm (thông thoáng), Thấp vào ban ngày (đông đúc)
    base_pattern = np.sin(x) 
    
    # Tạo hiệu ứng giờ cao điểm (Rush Hour Dips)
    # Tạo chu kỳ 24h cho từng bước
    hours = np.tile(np.linspace(0, 24, 24 * 60 // FREQ_MIN), DAYS)
    
    # Hàm mô phỏng tắc nghẽn (Tốc độ giảm mạnh tại giờ 8h và 18h)
    rush_hour_effect = np.exp(-0.5 * (hours - 8)**2) + np.exp(-0.5 * (hours - 18)**2)
    
    # --- A. SINH TỐC ĐỘ (SPEED) ---
    # Tốc độ lý tưởng ~ 50km/h, Tốc độ tắc đường ~ 10-15km/h
    # Công thức: Base - Tắc nghẽn + Nhiễu
    speed_data = np.zeros((n_steps, n_nodes))
    
    for i in range(n_nodes):
        # Mỗi nút có một chút biến động riêng (Random offset)
        node_bias = np.random.normal(0, 5) 
        
        # Tốc độ theo thời gian
        speed_series = 40 - (rush_hour_effect * 25) + node_bias + np.random.normal(0, 2, n_steps)
        
        # Giới hạn tốc độ hợp lý (0 - 80 km/h)
        speed_data[:, i] = np.clip(speed_series, 5, 50)

    # --- B. SINH LƯU LƯỢNG (FLOW) ---
    # Lưu lượng thường ngược chiều hoặc tương quan phức tạp với tốc độ
    # Giờ cao điểm: Lưu lượng cao (nhu cầu cao) dù tốc độ thấp
    flow_data = np.zeros((n_steps, n_nodes))
    
    for i in range(n_nodes):
        # Nhu cầu đi lại cao vào giờ cao điểm
        demand_pattern = rush_hour_effect * 100  # Cơ sở 100 xe
        base_flow = 20 + demand_pattern + np.random.poisson(5, n_steps)
        
        flow_data[:, i] = np.clip(base_flow, 0, 500)
        
    return speed_data, flow_data

In [28]:
# --- Hàm mô phỏng thời tiết ---
def generate_weather(n_steps):
    # Nhiệt độ: Thấp nhất 18 độ (đêm), Cao nhất 30 độ (trưa)
    hours = np.tile(np.linspace(0, 24, 24 * 60 // FREQ_MIN), DAYS)
    temp = 24 + 6 * np.sin((hours - 9) * 2 * np.pi / 24) + np.random.normal(0, 0.5, n_steps)
    
    # Thời tiết: 0=Clear, 1=Rain (Giả sử mưa ngẫu nhiên 10% thời gian)
    condition = np.random.choice([0, 1], size=n_steps, p=[0.9, 0.1])
    
    # Nếu mưa (condition=1), giảm tốc độ giao thông một chút (Logic thực tế)
    return temp, condition



In [29]:
# Gọi hàm sinh dữ liệu
speed_matrix, flow_matrix = generate_traffic_pattern(TOTAL_STEPS, num_nodes)
temp_array, rain_array = generate_weather(TOTAL_STEPS)

# Điều chỉnh Tốc độ khi có Mưa (Mưa làm giảm tốc độ 10-20%)
for t in range(TOTAL_STEPS):
    if rain_array[t] == 1:
        speed_matrix[t, :] *= np.random.uniform(0.8, 0.9)

In [30]:
# ==========================================
# 4. LƯU DỮ LIỆU (FILE OUTPUT)
# ==========================================
print("💾 Đang lưu dữ liệu xuống file...")

# 1. Ma trận kề (Dùng cho cấu trúc GCN)
np.save("hanoi_adj_matrix.npy", adj_matrix)

# 2. Dữ liệu Feature (Node Features) - Tốc độ
df_speed = pd.DataFrame(speed_matrix, index=time_index, columns=node_ids)
df_speed.to_csv("hanoi_speed.csv")

# 3. Dữ liệu Feature - Lưu lượng
df_flow = pd.DataFrame(flow_matrix, index=time_index, columns=node_ids)
df_flow.to_csv("hanoi_flow.csv")

# 4. Dữ liệu Ngoài (Weather) - Gắn vào từng bước thời gian
df_weather = pd.DataFrame({
    'Temperature': temp_array,
    'Rain': rain_array
}, index=time_index)
df_weather.to_csv("hanoi_weather.csv")

# 5. Dữ liệu Meta (Vị trí các nút để vẽ bản đồ sau này)
nodes_gdf[['y', 'x']].to_csv("hanoi_nodes_location.csv")

print("\n🎉 HOÀN TẤT! Bạn đã có bộ dữ liệu sau:")
print("1. hanoi_speed.csv       (Tốc độ theo thời gian)")
print("2. hanoi_flow.csv        (Lưu lượng theo thời gian)")
print("3. hanoi_weather.csv     (Thời tiết: Nhiệt độ, Mưa)")
print("4. hanoi_adj_matrix.npy  (Ma trận kề của Graph)")
print("5. hanoi_nodes_location.csv (Tọa độ Latitude/Longitude của các nút)")

💾 Đang lưu dữ liệu xuống file...

🎉 HOÀN TẤT! Bạn đã có bộ dữ liệu sau:
1. hanoi_speed.csv       (Tốc độ theo thời gian)
2. hanoi_flow.csv        (Lưu lượng theo thời gian)
3. hanoi_weather.csv     (Thời tiết: Nhiệt độ, Mưa)
4. hanoi_adj_matrix.npy  (Ma trận kề của Graph)
5. hanoi_nodes_location.csv (Tọa độ Latitude/Longitude của các nút)


In [44]:
import os

# Tên file chính xác (được in ra ở đầu code)
# Bạn phải thay thế bằng tên file được tạo ra khi bạn bắt đầu chạy 7 tiếng!
OUTPUT_FILE_NAME = "hanoi_traffic_FULL_SCAN_20251127_1211.csv" 

if os.path.exists(OUTPUT_FILE_NAME):
    # Lấy kích thước file tính bằng Megabytes
    file_size_mb = os.path.getsize(OUTPUT_FILE_NAME) / (1024 * 1024)
    print(f"✅ DỮ LIỆU ĐÃ LƯU!")
    print(f"Kích thước File: {file_size_mb:.2f} MB")
else:
    print("⚠️ LỖI: File không tồn tại trên ổ đĩa. Hãy kiểm tra lại file browser (Refresh).")

✅ DỮ LIỆU ĐÃ LƯU!
Kích thước File: 0.18 MB
